In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

model = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0
)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x107fc0ec0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x107fc1be0>, model_name='qwen/qwen3-32b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from typing_extensions import TypedDict
from typing import Literal
import random

class State(TypedDict):
    graph_info: str


def start_play(state: State):
    print("start_play has been called")
    return {"graph_info": state["graph_info"] + "Starting to play. "}


def cricket(state: State):
    print("cricket has been called")
    return {"graph_info": state["graph_info"] + "Playing cricket. "}


def badminton(state: State):
    print("badminton has been called")
    return {"graph_info": state["graph_info"] + "Playing badminton. "}


def random_play(state: State) -> Literal["cricket", "badminton"]:
    graph_info = state["graph_info"]

    if random.random() > 0.5:
        return "cricket"
    else:
        return "badminton"


from IPython.display import display, Image
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

graph.add_node("start_play", start_play)
graph.add_node("cricket", cricket)
graph.add_node("badminton", badminton)

graph.add_edge(START, "start_play")

graph.add_conditional_edges(
    "start_play", 
    random_play,
    {
        "cricket": "cricket",
        "badminton": "badminton"
    }
)

graph.add_edge("cricket", END)
graph.add_edge("badminton", END)

graph_builder = graph.compile()

print(graph_builder.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start_play(start_play)
	cricket(cricket)
	badminton(badminton)
	__end__([<p>__end__</p>]):::last
	__start__ --> start_play;
	start_play -.-> badminton;
	start_play -.-> cricket;
	badminton --> __end__;
	cricket --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
graph_builder.invoke({"graph_info": "Hey my name is Tejas "})

start_play has been called
cricket has been called


{'graph_info': 'Hey my name is Tejas Starting to play. Playing cricket. '}